In [ ]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))


SOC_ORDER = ["90%", "50%", "10%"]

# 实验预期值
CYCLE_ACTIVE_HOUR = 3.0
PAUSE_AFTER_CYCLE_HOUR = 4.5
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR。
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0
CYCLE_ACTIVE_LIMIT_MIN = CYCLE_ACTIVE_LIMIT_HOUR * 60

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current_level",
    "Zustand",
    "ID"
]

Note: you may need to restart the kernel to use updated packages.


In [29]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [ ]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df,debug=False):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间与电流处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
    df_td.groupby("File")["Time"]
    .diff()
    .dt.total_seconds()
    .div(3600)
)
    
    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ).dt.total_seconds().div(60)

    df_td["Current_level"] = df_td["Current"]

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    df_td["Zustand"] = df_td["Zustand"].str.upper()

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 只保留 CHA / DCH
    # -----------------------------
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)

    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        current_values = group["Current"]

        # 如果首个测量点 Current_level == 0，则忽略首点
        if len(group) >= 2 and group["Current_level"].iloc[0] == 0:
            current_values = current_values.iloc[1:]

        # 有效点太少时不做标准差剔除
        if len(current_values) <= 1:
            return False

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    bad_segments = (
        pulse_sequence
        .groupby(["File", "pulse_segment_id"])
        .filter(is_bad_current_segment)
        [["File", "pulse_segment_id"]]
        .drop_duplicates()
    )

    pulse_sequence = pulse_sequence.merge(
        bad_segments.assign(to_drop=True),
        on=["File", "pulse_segment_id"],
        how="left"
    )

    pulse_sequence = (
        pulse_sequence[pulse_sequence["to_drop"].isna()]
        .drop(columns="to_drop")
    )

    # -----------------------------
    # 7. 每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def pick_pulse_row(group):
        group = group.sort_values("Time")

        # 如果该脉冲段第一个测量点 Current_level 为 0，
        # 就改用第二个测量点记录
        if group["Current_level"].iloc[0] == 0 and len(group) >= 2:
            return group.iloc[1]

        return group.iloc[0]

    pulse_sequence = (
        pulse_sequence
        .groupby(["File", "pulse_segment_id"], group_keys=False)
        .apply(pick_pulse_row)
        .reset_index(drop=True)
    )

    # -----------------------------
    # 8. 剔除每个 cycle 前 1h 内 pulse 且状态为DCH的序列
    # -----------------------------
    drop_early_dch_mask = (
    (pulse_sequence["time_from_cycle_start_min"] <= REMOVE_PULSE_BEFORE_MIN)
    & pulse_sequence["Zustand"].str.startswith("DCH", na=False)
    )

    pulse_sequence = pulse_sequence[~drop_early_dch_mask].copy()

    # -----------------------------
    # 9. 最后剔除 DCH 且 abs(Current_level) 接近 1.5A
    # -----------------------------
    pulse_sequence = pulse_sequence[
        ~(
            pulse_sequence["Zustand"].str.startswith("DCH", na=False)
            & np.isclose(
                pulse_sequence["Current_level"].abs(),
                1.5,
                atol=0.08
            )
        )
    ].copy()

    # -----------------------------
    # 10. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

    return pulse_sequence

pulse_sequence = build_time_diff_sequence(df, debug=True)

display(pulse_sequence)

10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}
